[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antfolk/BMEN35_2026/blob/main/Session1/BMEN35_Ex1_03_KNN_step_by_step.ipynb)

# k-nearest neighbours (KNN), one step at a time

In this notebook we will calculate KNN predictions ourselves. We will look at the data, split it, scale the features, calculate distances, sort them, and let the nearest flowers vote. We will write the distance and voting code explicitly so you can see what happens at each step.

We will **not** use scikit-learn's KNN classifier in this walkthrough. Scikit-learn will help us load Iris, split the observations and scale the features, but we will write the KNN prediction code ourselves.

Run one cell at a time. Look at the table and the printed distances before moving on. KNN is a simple idea, but it is useful to see how each calculation contributes to the final class.

## 1. Import the libraries

NumPy will hold the arrays and help with arithmetic. Pandas will display the observations as a table, and Matplotlib will draw them. We will use scikit-learn to load the Iris data, split it and scale its features. There is no call to a scikit-learn KNN model in this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 2. Load the Iris data

The Iris dataset is a small, well-known dataset with measurements for three types of iris flowers. Each row is one flower. The four columns are sepal length, sepal width, petal length and petal width. The target number tells us the species.

The dataset is useful for learning because it is small and easy to plot. It is not biomedical data, and the results should not be generalized to another problem.

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target
number_of_classes = len(iris.target_names)

Let's print the names and dimensions before doing anything else. `X` has one row for each flower and one column for each measurement. `y` has one class label for every row in `X`.

In [ ]:
print('Feature names:', iris.feature_names)
print('Species names:', iris.target_names)
print('Shape of X:', X.shape)
print('Shape of y:', y.shape)
print('Class codes:', np.unique(y))

## 3. Look at the observations

Let's put a few rows into a table. The target codes are added as a separate column so we can connect the features with the class label. We will also display the class names; code 0, 1 or 2 is only a number until we check what it means.

In [ ]:
iris_table = pd.DataFrame(X, columns=iris.feature_names)
iris_table['class code'] = y
iris_table['species'] = [iris.target_names[class_code] for class_code in y]
display(iris_table.head(10))

print('Number of flowers in each class:')
for class_code, species_name in enumerate(iris.target_names):
    number_of_flowers = np.sum(y == class_code)
    print(species_name, ':', number_of_flowers)

## 4. Plot two of the four measurements

We cannot show all four feature columns in one ordinary scatter plot. Here we use petal length on the x-axis and petal width on the y-axis. Each point is one flower, and the color shows its species.

The plot is only one view of the data. The KNN distance below will use all four features.

In [ ]:
colors = ['red', 'green', 'blue']

for class_code, species_name in enumerate(iris.target_names):
    # Select the rows belonging to this species.
    rows_for_species = y == class_code

    plt.scatter(
        X[rows_for_species, 2],
        X[rows_for_species, 3],
        color=colors[class_code],
        label=species_name,
        edgecolor='black',
        alpha=0.75
    )

plt.xlabel('Petal length (cm)')
plt.ylabel('Petal width (cm)')
plt.title('Iris data: two of the four features')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

### Stop and look at the plot

1. Which species looks easiest to separate using these two features?
2. Are any species mixed together in this view?
3. What might be different if we also looked at sepal length and sepal width?

We are going to use all four measurements for KNN, even though we can only see two of them in this plot.

## 5. Make training, validation and test sets

KNN keeps the training observations and compares a new observation with them. We will use a training set for those comparisons, a validation set to compare a few values of `k`, and a test set for one final check.

The first split keeps 60% for training and 40% aside. We then divide that remaining 40% equally between validation and test. `stratify` keeps the species proportions similar in all three sets.

In [ ]:
X_train, X_remaining, y_train, y_remaining = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=42,
    stratify=y
)

X_validation, X_test, y_validation, y_test = train_test_split(
    X_remaining,
    y_remaining,
    test_size=0.50,
    random_state=42,
    stratify=y_remaining
)

print('Training flowers:', len(X_train))
print('Validation flowers:', len(X_validation))
print('Test flowers:', len(X_test))

## 6. Scale the features

KNN uses distances. If one feature has numbers much larger than another feature, that feature can dominate the distance even if it is not more important.

`StandardScaler` subtracts the training mean and divides by the training standard deviation for each feature. We fit the scaler on `X_train` only. Then we use those same training values to scale the validation and test sets. This keeps information from those held-out rows out of preprocessing.

In [ ]:
scaler = StandardScaler()

# Learn the mean and standard deviation from training flowers only.
X_train_scaled = scaler.fit_transform(X_train)

# Apply the training-set scaling to the validation and test flowers.
X_validation_scaled = scaler.transform(X_validation)
X_test_scaled = scaler.transform(X_test)

print('Scaled training shape:', X_train_scaled.shape)
print('Scaled validation shape:', X_validation_scaled.shape)
print('Scaled test shape:', X_test_scaled.shape)
print('Means of the scaled training columns:', X_train_scaled.mean(axis=0).round(3))

## 7. Choose one flower to classify

Let's start with one flower from the validation set. `query` is the flower we want to classify. `training_flower` is one flower from the training set. We will calculate the distance between these two rows first, before repeating the same calculation for all training flowers.

In [ ]:
query = X_validation_scaled[0]
training_flower = X_train_scaled[0]

print('Query flower:', query.round(3))
print('First training flower:', training_flower.round(3))
print('True species of query:', iris.target_names[y_validation[0]])

## 8. Calculate one distance in small steps

Euclidean distance is the straight-line distance between two points. For four features we subtract the feature values, square each difference, add the squared differences and take the square root:

$$
d(\mathbf{x},\mathbf{z})=\sqrt{(x_1-z_1)^2+(x_2-z_2)^2+(x_3-z_3)^2+(x_4-z_4)^2}.
$$

Let's keep each part of the calculation in a different variable. This is longer than writing one formula, but it makes the calculation easier to follow.

First subtract the four feature values. The result is one difference for each feature. Some differences can be negative; that is fine at this stage.

In [ ]:
difference = query - training_flower
print('Difference for each feature:', difference.round(3))

Now square each difference. This makes every contribution non-negative and gives larger differences a larger effect on the distance.

In [ ]:
squared_difference = difference ** 2
print('Squared difference for each feature:', squared_difference.round(3))

Add the four squared differences. This is the squared Euclidean distance. We do not take the square root until the next cell.

In [ ]:
sum_of_squared_differences = squared_difference.sum()
print('Sum of squared differences:', round(sum_of_squared_differences, 3))

Finally, take the square root. This gives the Euclidean distance between the two scaled flowers.

In [ ]:
distance = np.sqrt(sum_of_squared_differences)
print('Distance:', round(distance, 3))

The distance is one number that summarizes the difference across all four scaled measurements. A smaller distance means the two flowers are more similar according to these features and this distance rule.

## 9. Calculate the distance to every training flower

Now we repeat the same steps for every flower in the training set. We will use a loop over the training rows. Inside the loop we subtract the query, square each feature difference, add the squared values and take the square root.

We store two things for every training flower: its distance from the query and its known class label.

In [ ]:
# Make one row for every training flower and two columns: distance and class.
distance_and_class = np.zeros((len(X_train_scaled), 2))

for training_row_number in range(len(X_train_scaled)):
    training_flower = X_train_scaled[training_row_number]

    # Calculate feature-by-feature differences from this training flower.
    difference = query - training_flower
    squared_difference = difference ** 2
    sum_of_squared_differences = squared_difference.sum()
    distance = np.sqrt(sum_of_squared_differences)

    # Store the distance and the training flower's known class code.
    distance_and_class[training_row_number, 0] = distance
    distance_and_class[training_row_number, 1] = y_train[training_row_number]

print('First five distances and class codes:')
print(distance_and_class[:5])

## 10. Sort the flowers by distance

At the moment the distances are in the same order as the training rows. We need the smallest distance first. `np.argsort` returns the row positions that would sort the distances from smallest to largest.

Let's use those positions to sort the distance-and-class table. Then we can look at the closest flowers and see their species.

In [ ]:
# Find the positions of the distances from smallest to largest.
sort_positions = np.argsort(distance_and_class[:, 0])

# Use those positions to sort the full rows (distance and class stay together).
sorted_distance_and_class = distance_and_class[sort_positions]

print('Five closest distances and class codes:')
print(sorted_distance_and_class[:5])

In [ ]:
# Turn the first five rows into a table that is easier to read.
nearest_five_classes = sorted_distance_and_class[:5, 1].astype(int)
nearest_five_distances = sorted_distance_and_class[:5, 0]

nearest_flowers_table = pd.DataFrame({
    'distance': nearest_five_distances,
    'class code': nearest_five_classes,
    'species': [iris.target_names[class_code] for class_code in nearest_five_classes]
})
display(nearest_flowers_table)

The closest training flower is the first row. The next four rows are the next closest flowers. We now have the information KNN needs: the nearest rows, their distances, and their known species.

## 11. Let one nearest neighbor decide (`k = 1`)

When `k = 1`, only the closest flower gets a vote. Let's take its class code from the first sorted row and convert that code back to a species name.

In [ ]:
k = 1
nearest_class_codes = sorted_distance_and_class[:k, 1].astype(int)

prediction_k1 = nearest_class_codes[0]
print('Class code predicted with k=1:', prediction_k1)
print('Species predicted with k=1:', iris.target_names[prediction_k1])
print('True species:', iris.target_names[y_validation[0]])

## 12. Let three neighbors vote (`k = 3`)

Now the three closest flowers vote. We will count the votes explicitly rather than asking a model to do it. Start with zero votes for each of the three species, then add one vote for each neighbor's class.

In [ ]:
k = 3
nearest_class_codes = sorted_distance_and_class[:k, 1].astype(int)
votes = np.zeros(number_of_classes, dtype=int)

for class_code in nearest_class_codes:
    votes[class_code] = votes[class_code] + 1

print('Neighbor class codes:', nearest_class_codes)
print('Votes by species:', votes)

prediction_k3 = np.argmax(votes)
print('Species predicted with k=3:', iris.target_names[prediction_k3])

`np.argmax` gives us the position of the largest vote count. In this example, that position is also the class code. If two classes receive the same number of votes, `argmax` returns the first one; this is one possible way to break a tie.

## 13. Try five neighbors

Let's repeat the vote count with the five nearest flowers. Compare it with the result for `k = 1` and `k = 3`. The distances have not changed; only the number of neighbors taking part in the vote has changed.

In [ ]:
k = 5
nearest_class_codes = sorted_distance_and_class[:k, 1].astype(int)
votes = np.zeros(number_of_classes, dtype=int)

for class_code in nearest_class_codes:
    votes[class_code] = votes[class_code] + 1

print('Neighbor class codes:', nearest_class_codes)
print('Votes by species:', votes)

prediction_k5 = np.argmax(votes)
print('Species predicted with k=5:', iris.target_names[prediction_k5])

### Stop and compare the votes

1. Did `k = 1`, `k = 3` and `k = 5` give the same prediction?
2. What changes when we increase `k`? What does not change?
3. Could a very large `k` make a prediction worse? Why?

## 14. Write the distance-and-voting steps as a function

We have now seen each part of the calculation. A function lets us repeat the same steps for another query flower. The function below still uses loops: it calculates the distance to each training row, sorts the distances, and counts the class votes.

Read the function from top to bottom. It returns one class code for one query row.

In [ ]:
def predict_one_knn(query_row, training_features, training_labels, k):
    # Make room to store one distance and one class for each training row.
    distance_and_class = np.zeros((len(training_features), 2))

    # Compare the query with each training observation.
    for training_row_number in range(len(training_features)):
        distance_squared = 0

        # Add the squared difference for each feature.
        for feature_number in range(training_features.shape[1]):
            difference = (
                query_row[feature_number]
                - training_features[training_row_number, feature_number]
            )
            distance_squared = distance_squared + difference ** 2

        # The square root turns the squared distance into Euclidean distance.
        distance = np.sqrt(distance_squared)
        distance_and_class[training_row_number, 0] = distance
        distance_and_class[training_row_number, 1] = training_labels[training_row_number]

    # Sort the training rows from closest to furthest away.
    sort_positions = np.argsort(distance_and_class[:, 0])
    nearest_rows = distance_and_class[sort_positions[:k]]
    nearest_class_codes = nearest_rows[:, 1].astype(int)

    # Count one vote for each of the nearest class labels.
    votes = np.zeros(number_of_classes, dtype=int)
    for class_code in nearest_class_codes:
        votes[class_code] = votes[class_code] + 1

    # Return the class with the most votes.
    predicted_class = np.argmax(votes)
    return predicted_class

Let's call the function for the same query and `k = 5`. It should give the same answer as the separate cells above. The function has not changed the KNN method; it has just put the repeated steps under one name.

In [ ]:
function_prediction = predict_one_knn(
    query_row=X_validation_scaled[0],
    training_features=X_train_scaled,
    training_labels=y_train,
    k=5
)

print('Prediction from the function:', iris.target_names[function_prediction])
print('True species:', iris.target_names[y_validation[0]])

## 15. Compare a few values of `k` on validation data

A value of `k` is a model setting. We will compare `k = 1, 3, 5, 11, 21` using the validation flowers. For each value, we predict each validation flower with our function and calculate the fraction classified correctly.

We use validation data to compare the settings. We will leave the test flowers alone until we have chosen a value of `k`.

In [ ]:
candidate_k_values = [1, 3, 5, 11, 21]
validation_accuracies = []

for k in candidate_k_values:
    validation_predictions = []

    # Predict one validation flower at a time with our own KNN function.
    for validation_row in X_validation_scaled:
        prediction = predict_one_knn(
            validation_row,
            X_train_scaled,
            y_train,
            k
        )
        validation_predictions.append(prediction)

    validation_predictions = np.array(validation_predictions)

    # Count correct predictions one at a time so we can see what accuracy means.
    number_correct = 0
    for row_number in range(len(y_validation)):
        if validation_predictions[row_number] == y_validation[row_number]:
            number_correct = number_correct + 1

    accuracy = number_correct / len(y_validation)
    validation_accuracies.append(accuracy)

    print(
        'k =', k,
        '| correct =', number_correct, 'of', len(y_validation),
        '| validation accuracy =', round(accuracy, 3)
    )

In [ ]:
plt.plot(candidate_k_values, validation_accuracies, marker='o')
plt.xlabel('Number of neighbors, k')
plt.ylabel('Validation accuracy')
plt.title('Compare k values using the validation set')
plt.xticks(candidate_k_values)
plt.ylim(0, 1.05)
plt.grid(alpha=0.25)
plt.show()

The accuracy can go up or down as `k` changes. The validation set helps us choose a setting, but it is still only a small set of flowers. We should not keep changing `k` after looking at the test results.

## 16. Evaluate the chosen value once on the test set

We will choose the `k` value with the highest validation accuracy, then use it once on the test set. This gives us a final check on flowers that were not used to fit the scaler or to compare the `k` settings.

In [ ]:
best_position = np.argmax(validation_accuracies)
best_k = candidate_k_values[best_position]
print('k selected from validation results:', best_k)

test_predictions = []
for test_row in X_test_scaled:
    prediction = predict_one_knn(
        test_row,
        X_train_scaled,
        y_train,
        best_k
    )
    test_predictions.append(prediction)

test_predictions = np.array(test_predictions)

# Count the correct test predictions one by one.
number_correct = 0
for row_number in range(len(y_test)):
    if test_predictions[row_number] == y_test[row_number]:
        number_correct = number_correct + 1

test_accuracy = number_correct / len(y_test)
print('Correct test predictions:', number_correct, 'of', len(y_test))
print('Test accuracy:', round(test_accuracy, 3))

## 17. Make a confusion matrix with loops

Accuracy tells us the fraction correct, but it does not show which species are confused. We can count the true class and predicted class for every test flower in a 3 by 3 matrix.

Rows in the matrix will be the true species. Columns will be the predicted species. We will fill the matrix one observation at a time.

In [ ]:
confusion = np.zeros((3, 3), dtype=int)

for row_number in range(len(y_test)):
    true_class = y_test[row_number]
    predicted_class = test_predictions[row_number]

    # Add one to the cell for this true/predicted class pair.
    confusion[true_class, predicted_class] = (
        confusion[true_class, predicted_class] + 1
    )

confusion_table = pd.DataFrame(
    confusion,
    index=[f'True {name}' for name in iris.target_names],
    columns=[f'Predicted {name}' for name in iris.target_names]
)
display(confusion_table)

The diagonal cells are correct predictions. Values away from the diagonal are mistakes. The small test set means that one flower can noticeably change the counts, so do not make a broad claim from this one split.

## 18. What did we do?

We looked at the data, scaled the features using the training set, calculated Euclidean distances, sorted them, counted votes, and repeated those steps for validation and test flowers. Scikit-learn's KNN implementation would do these calculations for us, but here we wrote them out so the method is visible.

### Check your understanding

1. In the distance formula, why do we square the feature differences?
2. Why do we sort the distances before choosing the nearest neighbors?
3. What changes when we increase `k` from 1 to 5?
4. Why did we fit the scaler using the training set only?
5. Why did we use validation data to choose `k` and leave test data for one final check?
6. What information does the confusion matrix show that accuracy does not?

The assignment notebook uses a different dataset. You will fit a library model there, but you should now be able to explain what the KNN code is doing under the hood.